# SisaKulkas - Langkah 1: Dataset dan Pelatihan Model YOLOv8

**Sebelum mulai:** Runtime > Change runtime type > **T4 GPU**.

Simpan API key Roboflow (yang baru, bukan yang sudah tersebar) di **Secrets** (ikon kunci di sidebar kiri Colab) dengan nama `ROBOFLOW_API_KEY`, lalu aktifkan *Notebook access*.

## 1. Instalasi

In [ ]:
!pip install -q roboflow ultralytics

## 2. Unduh dataset

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("seongmin").project("food-ingredients-pnisb")
dataset = project.version(1).download("yolov8")
print("Lokasi dataset:", dataset.location)

## 3. Periksa isi dataset

Bagian ini penting: daftar kelas menentukan bahan apa saja yang bisa dikenali aplikasi, dan menjadi acuan untuk membuat database resep.

In [ ]:
import yaml, glob, os
from collections import Counter

with open(f"{dataset.location}/data.yaml") as f:
    cfg = yaml.safe_load(f)
names = cfg['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names)]
print(f"Jumlah kelas: {len(names)}")
print(names)

for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{dataset.location}/{split}/images/*")
    print(f"{split}: {len(imgs)} gambar")

# Jumlah objek per kelas (data train)
cnt = Counter()
for lf in glob.glob(f"{dataset.location}/train/labels/*.txt"):
    for line in open(lf):
        if line.strip():
            cnt[int(line.split()[0])] += 1
for i, n in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f"{names[i]:25s} {n}")

## 4. Latih model

Mulai dari `yolov8n` (paling ringan). Jika akurasi kurang, coba `yolov8s.pt` atau tambah epoch.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="sisakulkas",
    name="v1",
)

## 5. Evaluasi

In [ ]:
metrics = model.val()
print("mAP50    :", metrics.box.map50)
print("mAP50-95 :", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall   :", metrics.box.mr)

## 6. Uji prediksi pada gambar contoh

In [ ]:
from IPython.display import Image, display

best = YOLO("sisakulkas/v1/weights/best.pt")
sample = glob.glob(f"{dataset.location}/test/images/*")[:3]
preds = best.predict(sample, conf=0.4, save=True)
for p in preds:
    display(Image(filename=os.path.join(p.save_dir, os.path.basename(p.path))))

## 7. Simpan model

File `best.pt` akan dipakai di aplikasi web (langkah berikutnya).

In [ ]:
from google.colab import files
files.download("sisakulkas/v1/weights/best.pt")